In [2]:
!pip install -q --force-reinstall \
    torch==2.11.0 \
    torchvision==0.26.0 \
    torchaudio==2.11.0 \
    --index-url https://download.pytorch.org/whl/cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 

In [1]:
import torch
import torchvision
import faiss
import numpy as np

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("FAISS:", faiss.__version__)


Torch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
NumPy: 2.5.2
FAISS: 1.15.0


In [2]:
!pip install -q sentence-transformers transformers sentencepiece


In [3]:
import torch
import torchvision
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("FAISS:", faiss.__version__)
print("All imports successful!")


Torch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
NumPy: 2.5.2
FAISS: 1.15.0
All imports successful!


In [4]:
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

print("Knowledge base created!")


Knowledge base created!


In [5]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(
    documents,
    convert_to_numpy=True
)

print("Document embeddings created!")
print("Embedding shape:", doc_embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Document embeddings created!
Embedding shape: (4, 384)


In [6]:
import faiss

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(doc_embeddings.astype("float32"))

print("FAISS index created!")
print("Number of documents:", index.ntotal)


FAISS index created!
Number of documents: 4


In [7]:
query = "What is RAG in AI?"

query_embedding = embed_model.encode(
    [query],
    convert_to_numpy=True
)

D, I = index.search(
    query_embedding.astype("float32"),
    k=2
)

retrieved_chunks = [documents[i] for i in I[0]]

print("Question:", query)
print("\nRetrieved documents:")

for chunk in retrieved_chunks:
    print("-", chunk)


Question: What is RAG in AI?

Retrieved documents:
- Python is a popular high-level programming language used in AI development.
- Retrieval-Augmented Generation combines document retrieval with text generation.


In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Combine retrieved documents
context = " ".join(retrieved_chunks)

# Create prompt
prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

# Load FLAN-T5
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Convert prompt to tokens
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

# Generate answer
outputs = model.generate(
    **inputs,
    max_new_tokens=60
)

# Convert tokens back to text
answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Question:", query)
print("\nRetrieved Context:")

for chunk in retrieved_chunks:
    print("-", chunk)

print("\nFinal Answer:")
print(answer)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Question: What is RAG in AI?

Retrieved Context:
- Python is a popular high-level programming language used in AI development.
- Retrieval-Augmented Generation combines document retrieval with text generation.

Final Answer:
combines document retrieval with text generation
